In [1]:
#Run this cell once to initialize the robot and start the navigation loop. 
#If you need to stop the robot, run the last cell to detach the camera and zero the motors.
import cv2
import numpy as np
import time
import json
import base64
import ipywidgets as widgets
from IPython.display import display
from urllib import request, error
from jetbot import Camera, Robot, bgr8_to_jpeg


In [2]:
# ── 1. Cloud API Configuration ──────────────────────────────────────
API_KEY = "Ub1KVwtGHHdLLKRzoxdG"
API_URL = "https://serverless.roboflow.com/kais-workspace-stbmo/workflows/detect-count-and-visualize-3"
CONFIDENCE_THRESHOLD = 0.5 

# ── 2. Camera & HSV Configuration (Dynamic Load) ────────────────────
import yaml
import numpy as np
try:
    with open("config.yaml", encoding="utf-8") as f_cfg:
        _config = yaml.safe_load(f_cfg)
    CAMERA_WIDTH = _config["camera"]["width"]
    CAMERA_HEIGHT = _config["camera"]["height"]
    TAPE_HSV_LOWER = np.array(_config["color"]["lower_hsv"])
    TAPE_HSV_UPPER = np.array(_config["color"]["upper_hsv"])
    print("Loaded configuration from config.yaml")
except Exception as _e_cfg:
    print("Could not load config.yaml, using defaults:", _e_cfg)
    CAMERA_WIDTH  = 320
    CAMERA_HEIGHT = 240
    TAPE_HSV_LOWER = np.array([15,  60,  60])
    TAPE_HSV_UPPER = np.array([45, 255, 255])

# ── 3. Motor Speeds & Timing ────────────────────────────────────────
FORWARD_SPEED = 0.18   # normal wander speed
REVERSE_SPEED = 0.15   # backing away from a hazard
TURN_SPEED    = 0.22   # in-place turn during avoidance
MAX_SPEED     = 0.25   # hard clamp on all motor commands

AVOID_REVERSE_TIME = 0.40   # seconds: phase 1 reverse
AVOID_TURN_TIME    = 0.55   # seconds: phase 2 turn
WANDER_COOLDOWN    = 0.50   # seconds: free drive after avoidance

# ── 4. Local Vision Thresholds (Reflexes) ───────────────────────────
BOTTOM_ROI_FRACTION   = 0.30   # bottom 30% for yellow tape
FRONT_ROI_TOP_FRAC    = 0.15   # obstacle zone starts at 15%
FRONT_ROI_BOTTOM_FRAC = 0.65   # obstacle zone ends at 65%

YELLOW_BOUNDARY_THRESHOLD = 1800   
OBSTACLE_EDGE_THRESHOLD   = 500    

CANNY_LOW  = 40
CANNY_HIGH = 120

# ── 5. Initialize Hardware & UI ─────────────────────────────────────
import cv2
from jetbot import Robot, Camera, bgr8_to_jpeg
import ipywidgets as widgets
from IPython.display import display

robot = Robot()
robot.stop()

try:
    camera = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
    print("Real camera initialized successfully.")
except Exception as e:
    print(f"Could not initialize real camera: {e}")
    print("Falling back to MockCamera...")
    try:
        from tests.mock_camera import MockCamera
        camera = MockCamera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
        print("MockCamera initialized successfully.")
    except Exception as e2:
        print(f"Failed to load MockCamera: {e2}. Creating dummy camera...")
        class DummyCamera:
            def __init__(self):
                self.value = np.zeros((CAMERA_HEIGHT, CAMERA_WIDTH, 3), dtype=np.uint8)
            def observe(self, *args, **kwargs): pass
            def unobserve_all(self, *args, **kwargs): pass
            def stop(self, *args, **kwargs): pass
        camera = DummyCamera()
        print("DummyCamera initialized successfully.")

image_widget = widgets.Image(format='jpeg', width=CAMERA_WIDTH,  height=CAMERA_HEIGHT)
mask_widget  = widgets.Image(format='jpeg', width=CAMERA_WIDTH,  height=CAMERA_HEIGHT)
state_label  = widgets.Label(value='State: STOPPED')
motor_label  = widgets.Label(value='Motors: L=0.00  R=0.00')
yellow_label = widgets.Label(value='Yellow: 0  (L=0 R=0)')
obs_label    = widgets.Label(value='Edges: 0  (L=0 R=0)')
turn_label   = widgets.Label(value='Turn dir: -')

display(widgets.VBox([
    widgets.HBox([image_widget, mask_widget]),
    widgets.VBox([state_label, motor_label, yellow_label, obs_label, turn_label])
]))

print(' Setup Complete. Camera, Motors, and UI Ready.')


Loaded configuration from config.yaml
Could not initialize real camera. Falling back to MockCamera.
Real camera initialized successfully.


 Setup Complete. Camera, Motors, and UI Ready.


In [3]:
#run once to initialize the Vision Brain & Motor Helpers. These functions handle the computer vision processing and motor control for the robot.
# ── Cloud Vision ──────────────────────────────────────────────────
def infer_frame(frame):
    """Sends frame to Roboflow."""
    ok, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok: return None
        
    image_b64 = base64.b64encode(enc.tobytes()).decode("utf-8")
    payload = {
        "api_key": API_KEY,
        "inputs": {"image": {"type": "base64", "value": image_b64}, "confidence": CONFIDENCE_THRESHOLD}
    }

    req = request.Request(
        url=API_URL, data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}, method="POST"
    )

    try:
        with request.urlopen(req, timeout=1.5) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        return None

# ── Local Vision ──────────────────────────────────────────────────
def detect_yellow_boundary(frame):
    h = frame.shape[0]
    roi_start = int(h * (1.0 - BOTTOM_ROI_FRACTION))
    roi = frame[roi_start:, :]

    hsv_roi  = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    roi_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    yellow_area  = int(np.sum(roi_mask > 0))
    mid          = roi_mask.shape[1] // 2
    yellow_left  = int(np.sum(roi_mask[:, :mid] > 0))
    yellow_right = int(np.sum(roi_mask[:, mid:] > 0))

    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_start:, :] = roi_mask

    return {
        'detected':     yellow_area >= YELLOW_BOUNDARY_THRESHOLD,
        'yellow_area':  yellow_area,
        'yellow_left':  yellow_left,
        'yellow_right': yellow_right,
        'yellow_mask':  full_mask,
    }

def detect_obstacle(frame):
    h, w    = frame.shape[:2]
    roi_top = int(h * FRONT_ROI_TOP_FRAC)
    roi_bot = int(h * FRONT_ROI_BOTTOM_FRAC)
    roi     = frame[roi_top:roi_bot, :]

    hsv_roi     = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    yellow_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray[yellow_mask > 0] = 0

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges   = cv2.Canny(blurred, CANNY_LOW, CANNY_HIGH)
    edges   = cv2.dilate(edges, None, iterations=1)

    mid        = edges.shape[1] // 2
    edge_left  = int(np.sum(edges[:, :mid] > 0))
    edge_right = int(np.sum(edges[:, mid:] > 0))
    edge_total = edge_left + edge_right

    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_top:roi_bot, :] = edges

    return {
        'detected':   edge_total >= OBSTACLE_EDGE_THRESHOLD,
        'edge_total': edge_total,
        'edge_left':  edge_left,
        'edge_right': edge_right,
        'mask':       full_mask,
    }

# ── Motor Helpers ─────────────────────────────────────────────────
def clamp(val, lo, hi):
    if val < lo: return lo
    if val > hi: return hi
    return val

def safe_stop():
    try:
        robot.left_motor.value  = 0.0
        robot.right_motor.value = 0.0
    except Exception: pass

def set_drive(left, right):
    robot.left_motor.value  = clamp(left,  -MAX_SPEED, MAX_SPEED)
    robot.right_motor.value = clamp(right, -MAX_SPEED, MAX_SPEED)

print('Vision Brain & Motor Helpers Loaded.')

Vision Brain & Motor Helpers Loaded.


In [4]:
#run also once The State Machine & Execution Loop
nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
wander_cooldown_end = 0.0
last_left_spd       = 0.0
last_right_spd      = 0.0

# Cloud specific states
last_cloud_check    = 0.0
cloud_predictions   = [] 

def execute(change):
    global nav_state, avoid_start_time, avoid_turn_right
    global wander_cooldown_end, last_left_spd, last_right_spd
    global last_cloud_check, cloud_predictions

    frame     = change['new']
    left_spd  = 0.0
    right_spd = 0.0

    try:
        now      = time.time()
        boundary = detect_yellow_boundary(frame)
        obstacle = detect_obstacle(frame)

        # ── 1. CLOUD GLANCE (Every 1.0 seconds while safe) ────────────
        if (now - last_cloud_check) > 1.0 and nav_state == 'WANDER':
            last_cloud_check = now
            result = infer_frame(frame.copy())
            if result and "outputs" in result and result["outputs"]:
                cloud_predictions = result["outputs"][0].get("predictions", {}).get("predictions", [])
            else:
                cloud_predictions = []

        # ── 2. STATE MACHINE (Reflexes) ──────────────────────────────
        if nav_state == 'STOPPED':
            pass   

        elif nav_state in ('AVOID_BOUNDARY', 'AVOID_OBSTACLE'):
            elapsed = now - avoid_start_time
            if elapsed < AVOID_REVERSE_TIME:
                left_spd, right_spd = -REVERSE_SPEED, -REVERSE_SPEED
            elif elapsed < AVOID_REVERSE_TIME + AVOID_TURN_TIME:
                if avoid_turn_right:
                    left_spd, right_spd = TURN_SPEED, -TURN_SPEED
                else:
                    left_spd, right_spd = -TURN_SPEED, TURN_SPEED
            else:
                wander_cooldown_end = now + WANDER_COOLDOWN
                nav_state = 'WANDER'
                left_spd, right_spd = FORWARD_SPEED, FORWARD_SPEED

        elif nav_state == 'WANDER':
            if now < wander_cooldown_end:
                left_spd, right_spd = FORWARD_SPEED, FORWARD_SPEED
            elif boundary['detected']:
                avoid_turn_right = boundary['yellow_left'] >= boundary['yellow_right']
                avoid_start_time = now
                nav_state        = 'AVOID_BOUNDARY'
                left_spd, right_spd = -REVERSE_SPEED, -REVERSE_SPEED
            elif obstacle['detected']:
                avoid_turn_right = obstacle['edge_left'] >= obstacle['edge_right']
                avoid_start_time = now
                nav_state        = 'AVOID_OBSTACLE'
                left_spd, right_spd = -REVERSE_SPEED, -REVERSE_SPEED
            else:
                left_spd, right_spd = FORWARD_SPEED, FORWARD_SPEED

        # ── 3. DRIVE COMMAND ─────────────────────────────────────────
        set_drive(left_spd, right_spd)

        # ── 4. UI UPDATE ─────────────────────────────────────────────
        annotated = frame.copy()
        h, w = annotated.shape[:2]
        
        # Draw Zones
        roi_y = int(h * (1.0 - BOTTOM_ROI_FRACTION))
        cv2.rectangle(annotated, (0, roi_y), (w - 1, h - 1), (0, 0, 255) if boundary['detected'] else (0, 220, 220), 2)
        obs_top = int(h * FRONT_ROI_TOP_FRAC)
        obs_bot = int(h * FRONT_ROI_BOTTOM_FRAC)
        cv2.rectangle(annotated, (0, obs_top), (w - 1, obs_bot), (0, 0, 255) if obstacle['detected'] else (255, 130, 0), 2)
        
        # Draw Cloud Predictions (Magenta)
        for pred in cloud_predictions:
            px, py, pw, ph = pred["x"], pred["y"], pred["width"], pred["height"]
            x1, y1 = int(px - pw / 2), int(py - ph / 2)
            x2, y2 = int(px + pw / 2), int(py + ph / 2)
            label = f'{pred["class"]} {pred["confidence"]:.2f}'
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 255), 2)
            cv2.putText(annotated, label, (x1, max(y1 - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 1)

        image_widget.value = bgr8_to_jpeg(annotated)

        # Draw Mask Widget
        combined = np.zeros((CAMERA_HEIGHT, CAMERA_WIDTH, 3), dtype=np.uint8)
        combined[boundary['yellow_mask'] > 0] = [0, 255, 255]
        combined[obstacle['mask'] > 0] = [0, 140, 255]
        mask_widget.value = bgr8_to_jpeg(combined)

        state_label.value  = 'State: ' + nav_state
        motor_label.value  = f'Motors L={left_spd:.3f}  R={right_spd:.3f}'
        yellow_label.value = f"Yellow: {boundary['yellow_area']}  (L={boundary['yellow_left']} R={boundary['yellow_right']})"
        obs_label.value    = f"Edges: {obstacle['edge_total']}  (L={obstacle['edge_left']} R={obstacle['edge_right']})"
        turn_label.value   = 'Turn dir: ' + ('RIGHT' if avoid_turn_right else 'LEFT')

    except Exception as e:
        safe_stop()
        print('[ERROR] ' + str(e))
        raise

print(' Execution Loop Ready.')

 Execution Loop Ready.


In [5]:
#Run this to set the robot to a safe state and stop the navigation loop
nav_state           = 'WANDER'
avoid_start_time    = 0.0
last_cloud_check    = 0.0
cloud_predictions   = [] 

# Attach Callback (This turns the engine on!)
camera.unobserve_all()
camera.observe(execute, names='value')
print(' Navigation Started. State:', nav_state)

 Navigation Started. State: WANDER


In [6]:
#stop the robot and detach the camera. This is a safe state to leave the robot in.
camera.unobserve_all()
safe_stop()
nav_state = 'STOPPED'
state_label.value = 'State: STOPPED'
motor_label.value = 'Motors: L=0.00  R=0.00'
print('EMERGENCY STOP — camera detached, motors zeroed.')

EMERGENCY STOP — camera detached, motors zeroed.


In [ ]:
"""
Width-based bottle-cap pickup — runs ON THE ROBOT (JETANK + JetBot base).

Pipeline:
  detect cap (Roboflow) -> steer to center -> drive forward -> stop when the
  cap's bounding-box WIDTH (pixels) reaches the calibrated grab distance ->
  run the arm pickup sequence.

The cap's pixel width grows as the robot nears it, so width is our distance
proxy. Calibrate once: place a cap exactly where the claw can grab it, and
record the width the camera sees at that distance.

Subcommands:
  python3 robot_cap_pickup.py calibrate   # measure + save grab-distance width
  python3 robot_cap_pickup.py run         # autonomous detect->approach->pickup
  python3 robot_cap_pickup.py pickup      # run the arm pickup sequence only (test)

NOTE: needs jetbot + SCSCtrl + network (Roboflow). Cloud inference is ~0.5-2s
per frame, so motion is pulsed (move a little, stop, re-detect). Tune the
SPEED / PULSE / TOLERANCE constants on the real robot.
"""

import base64
import json
import os
import sys
import time
from urllib import request, error

import cv2

# --- robot libs (only available on the robot) ---
from jetbot import Robot, Camera
from src.SCSCtrl import TTLServo

# ==================== config ====================
API_KEY = "Ub1KVwtGHHdLLKRzoxdG"
API_URL = "https://serverless.roboflow.com/kais-workspace-stbmo/workflows/detect-count-and-visualize-3"
CONFIDENCE_THRESHOLD = 0.4     # lower than detection-only: catch caps from farther away
TARGET_CLASS = "bottle cap"

CAMERA_W = 224
CAMERA_H = 224

# Approach / steering (jetbot Robot speeds are 0.0-1.0). Tune on hardware.
FORWARD_SPEED = 0.12
TURN_SPEED = 0.12
PULSE_TIME = 0.15              # seconds per move pulse, then stop + re-detect
CENTER_TOLERANCE_PX = 25       # |cap_x - center| under this = "centered enough"
SEARCH_TURN_PULSE = 0.12       # rotate speed when no cap is visible

# Calibration: width (px) at which the cap is grabbable. Set by `calibrate`.
CALIB_FILE = "cap_calib.json"
DEFAULT_TARGET_WIDTH = 120     # fallback if no calibration file yet (PLACEHOLDER — calibrate!)
CALIB_SAMPLES = 10             # detections averaged during calibration
CALIB_COUNTDOWN = 5            # seconds to position the cap before sampling

# --- arm constants (from Joaquin-Test2.ipynb) ---
CLAW_SERVO_ID = 4
CLAW_OPEN = -10
CLAW_CLOSED = -75
ARM_HOME_X, ARM_HOME_Y = 130, 20
ARM_DOWN_X, ARM_DOWN_Y = 150, -45
DROP_X, DROP_Y = 130, 20       # tune physically for the container


# ==================== Roboflow ====================
def infer_frame(frame):
    ok, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError("Failed to JPEG-encode frame")
    image_b64 = base64.b64encode(enc.tobytes()).decode("utf-8")
    payload = {
        "api_key": API_KEY,
        "inputs": {
            "image": {"type": "base64", "value": image_b64},
            "confidence": CONFIDENCE_THRESHOLD,
        },
    }
    req = request.Request(
        url=API_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", "Accept": "application/json",
                 "User-Agent": "python-urllib/3"},
        method="POST",
    )
    with request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode("utf-8"))


def extract_predictions(result):
    outputs = result.get("outputs", [])
    if not outputs:
        return []
    preds = outputs[0].get("predictions", {})
    if isinstance(preds, dict):
        preds = preds.get("predictions", [])
    return preds or []


def detect_caps(frame):
    """Return list of cap predictions, or [] on any error."""
    try:
        result = infer_frame(frame)
    except (error.HTTPError, error.URLError) as e:
        print(f"Inference error: {e}")
        return []
    except Exception as e:
        print(f"Inference failed: {e}")
        return []
    return [p for p in extract_predictions(result) if p["class"] == TARGET_CLASS]


def pick_target(caps):
    """Choose the cap to go for: the nearest = largest bounding-box width."""
    if not caps:
        return None
    return max(caps, key=lambda p: p["width"])


# ==================== arm ====================
def claw_open():
    TTLServo.servoAngleCtrl(CLAW_SERVO_ID, CLAW_OPEN, 1, 150)


def claw_close():
    TTLServo.servoAngleCtrl(CLAW_SERVO_ID, CLAW_CLOSED, 1, 150)


def arm_to(x, y, speed=0.8):
    TTLServo.xyInputSmooth(x, y, speed)


def arm_home():
    arm_to(ARM_HOME_X, ARM_HOME_Y)


def pickup_sequence():
    """home -> open -> down -> close -> home (from Joaquin-Test2)."""
    print("PICKUP: starting sequence")
    arm_home();        time.sleep(2)
    claw_open();       time.sleep(1)
    arm_to(ARM_DOWN_X, ARM_DOWN_Y); time.sleep(2)
    claw_close();      time.sleep(1)
    arm_home();        time.sleep(2)
    print("PICKUP: complete")


def drop_sequence():
    """home -> over container -> release -> home."""
    print("DROP: starting")
    arm_home();        time.sleep(2)
    arm_to(DROP_X, DROP_Y); time.sleep(2)
    claw_open();       time.sleep(1)
    arm_home();        time.sleep(2)
    print("DROP: complete")


# ==================== calibration ====================
def load_target_width():
    if os.path.exists(CALIB_FILE):
        try:
            return float(json.load(open(CALIB_FILE))["target_width"])
        except Exception:
            pass
    print(f"No {CALIB_FILE} -> using DEFAULT_TARGET_WIDTH={DEFAULT_TARGET_WIDTH} "
          "(run `calibrate` for accuracy).")
    return float(DEFAULT_TARGET_WIDTH)


def mode_calibrate(camera):
    print("CALIBRATE: place a cap exactly where the claw can grab it "
          "(arm at ARM_DOWN reach).")
    for s in range(CALIB_COUNTDOWN, 0, -1):
        print(f"  sampling in {s}...")
        time.sleep(1)

    widths = []
    while len(widths) < CALIB_SAMPLES:
        caps = detect_caps(camera.value.copy())
        target = pick_target(caps)
        if target is None:
            print("  no cap seen — adjust position")
            time.sleep(0.5)
            continue
        widths.append(target["width"])
        print(f"  sample {len(widths)}/{CALIB_SAMPLES}: width={target['width']:.0f}px")

    target_width = sorted(widths)[len(widths) // 2]  # median, robust to outliers
    json.dump({"target_width": target_width}, open(CALIB_FILE, "w"))
    print(f"\nCALIBRATED: grab width = {target_width:.0f}px  -> saved to {CALIB_FILE}")


# ==================== autonomous run ====================
def pulse(robot, action, speed):
    """Do a short timed move, then stop (pulse control for slow inference loop)."""
    getattr(robot, action)(speed)
    time.sleep(PULSE_TIME)
    robot.stop()


def mode_run(camera, robot):
    target_width = load_target_width()
    center_x = CAMERA_W / 2
    print(f"RUN: approach until cap width >= {target_width:.0f}px. Ctrl-C to stop.\n")

    while True:
        frame = camera.value.copy()
        caps = detect_caps(frame)
        target = pick_target(caps)

        if target is None:
            # no cap -> rotate a little to search
            print("search: no cap, rotating")
            pulse(robot, "right", SEARCH_TURN_PULSE)
            continue

        cx, width = target["x"], target["width"]
        err = cx - center_x
        print(f"cap: x={cx:.0f} width={width:.0f}px err={err:+.0f}")

        if width >= target_width:
            robot.stop()
            print(">>> AT GRAB DISTANCE — picking up")
            pickup_sequence()
            # drop_sequence()   # uncomment when the container position is tuned
            print("Done. Stopping. (loop ends — re-run for the next cap.)")
            break
        elif abs(err) > CENTER_TOLERANCE_PX:
            pulse(robot, "right" if err > 0 else "left", TURN_SPEED)  # center first
        else:
            pulse(robot, "forward", FORWARD_SPEED)                    # then advance


# ==================== dispatch ====================
def main():
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"

    if mode == "pickup":          # arm-only test, no camera/drive
        pickup_sequence()
        return

    camera = Camera.instance(width=CAMERA_W, height=CAMERA_H)
    robot = Robot()
    try:
        if mode == "calibrate":
            mode_calibrate(camera)
        elif mode == "run":
            mode_run(camera, robot)
        else:
            print(__doc__)
    except KeyboardInterrupt:
        print("\nInterrupted.")
    finally:
        robot.stop()
        camera.stop()
        print("Robot + camera stopped.")


if __name__ == "__main__":
    main()

